# Notebook 05: Modeling

## Purpose
This notebook answers the final piece of my research question with a model: how much does each factor a team controls actually matter to a race result? I predict finishing position from my project features, then use the model as a measuring instrument to size the levers my stakeholder is choosing between: driver and car performance (via qualifying position) versus pit stop execution and strategy.

This is a retrospective modeling exercise. I am measuring which historical variables best explain finishing position, not building a tool to predict next Sunday's race. That framing matters for how the features are built and how the results should be read.


## The setup
- **Target:** finishing position. This is a regression problem.
- **Metric:** RMSE, which reads as "how many finishing positions off, on average." Lower is better.
- **Legal features:** grid, avg_pit_stop_duration, constructor, circuit, and era. Grid and circuit are known before the race. Pit stop duration is a season-level summary, which makes this a retrospective measuring model, not a live pre-race predictor (see caveats in the final section).
- **Banned for leakage:** position_change (it is calculated from the finishing position, so it contains the answer) and teammate_gap (built from season points, which come from finishes, and it lives at the wrong grain, one value per team season instead of per race).

## The model progression
1. **Baseline:** a dummy model that always guesses the average finish. This is the bar any real model must beat.
2. **Simple model:** a shallow decision tree using only grid and pit stop duration.
3. **Tuned model:** Gradient Boosting on all features, with hyperparameters validated by GridSearchCV using 5-fold cross validation on the training set only.

## What to expect
I predicted the outcome before running anything: the tuned model would beat the simple one by only a small margin, because my statistical analysis in notebook 04 already showed grid explains most of the finish (correlation of about 0.75). That prediction held. The small gain is not a failure, it is the finding. The real payoff of this notebook is the feature importance breakdown, which quantifies how much pit execution and other factors add on top of qualifying position.

## Data note
The modeling frame is 4,990 rows: finished races on a real grid slot from 2011 to 2024. The 2010 season is excluded because the dataset contains no 2010 pit stop data, and filling those values would fabricate the exact signal I am measuring.

In [1]:
# 05 Modeling: setup
# Predicting finishing position from features known before the race ends.
# Regression problem, metric is RMSE (how many positions off, on average).

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
# Load the processed tables (read-only, same rule as notebook 04)
results = pd.read_csv('../data/processed/results.csv')
avg_pit_stop = pd.read_csv('../data/processed/avg_pit_stop.csv')
races = pd.read_csv('../data/clean/races.csv')


## Setup: building a leakage-free modeling frame

### The legal feature set
- A feature is only allowed if I would know its value BEFORE the race finished. This is the leakage rule: I must not use finish information to predict the finish.
- **Allowed (5 features):** grid, avg_pit_stop_duration, constructorId, circuitId, era.
- **Banned for leakage:** position_change, because it is grid minus finishing position, so it literally contains the answer. It was my headline driver-skill proxy in every other notebook, but as a model feature it would let the model cheat. I also left out teammate_gap, because it is built from season points (which come from finishes) and it is the wrong grain (one value per team-season, not per race).

### Building the frame
- I started from finished races on a real grid slot (dropped DNFs and pit lane starts), the same clean rule as my stats notebook, which gives 5,337 rows.
- I merged in avg_pit_stop_duration by constructor and year.
- I merged in circuitId by race, since circuitId lives in the races table, not results.

### Missing data decision (drop, not fill)
- After merging, 347 rows had a null avg_pit_stop_duration. I checked where they came from: all 347 are from 2010, the one season this dataset has no pit stop data for.
- This is structural missingness, not random gaps. Filling those rows with something like a median would invent pit crew speed for a season that has none, which would corrupt the exact strategy signal I am trying to measure.
- So I dropped them rather than impute. I also dropped them BEFORE the train/test split, so that every model (baseline, simple, tuned) trains and tests on the exact same rows and the RMSE comparison stays apples to apples.
- Frame after dropping: 4,990 rows.

### Encoding the categorical features
- Three of my five features are categorical: constructorId, circuitId, and era. I one-hot encoded all three (each category becomes its own 0/1 column).
- constructorId and circuitId are stored as integer IDs, but those integers are name tags, not quantities. Left raw, a model would invent a fake ordering (it would think constructor 131 is "more than" constructor 9, which is meaningless). One-hot removes that.
- era is text, so the model cannot ingest it at all until it is numeric. era does have a genuine time order, so I could have encoded it 1 to 4, but I chose one-hot on purpose so the model does not assume the eras are evenly spaced or that a later era means "more" of something. I let each era stand on its own.
- After encoding, my feature matrix is 4,990 rows by 64 columns (2 numeric, 4 era, 23 constructor, 35 circuit).

### Leakage firewall check
- Before training anything, I verified my feature matrix contained no target-derived columns: no position, no position_change, no points, no IDs, and nothing non-numeric. The check came back clean.
- This matters because a model with leakage produces a beautiful score and is worthless. Running this check by hand, before any model exists, is the guardrail.

### Train/test split (before any model)
- I split 80/20 into 3,992 training rows and 998 test rows, with a fixed random_state so the split is reproducible.
- The test set is held back as unseen "new races." No model touches it during training; it is only used for final scoring. Splitting before building any model is what keeps the evaluation honest.

In [3]:
# Build the modeling frame.
# Target = position (finishing position). Every feature must be knowable
# BEFORE the race finishes, so position_change is banned (it contains position).

# Start from finished races on a real grid slot, same clean rule as the stats notebook
model_df = results[(results['position'].notna()) & (results['grid'] != 0)].copy()

# Attach avg_pit_stop_duration by team and year
model_df = model_df.merge(avg_pit_stop, on=['constructorId', 'year'], how='left')

# Attach circuitId by race. circuitId lives in races.csv, not results,
# so without this merge the frame is missing a feature the model needs.
model_df = model_df.merge(races[['raceId', 'circuitId']], on='raceId', how='left')

# The five legal features and the target
feature_cols = ['grid', 'avg_pit_stop_duration', 'constructorId', 'circuitId', 'era']
target_col = 'position'

In [4]:
# Verify the merges actually populated, not silently filled NaN
print('rows in model_df:', len(model_df))
print('circuitId nulls:', model_df['circuitId'].isna().sum())
print('avg_pit_stop_duration nulls:', model_df['avg_pit_stop_duration'].isna().sum())
print(model_df[feature_cols].head())

rows in model_df: 5337
circuitId nulls: 0
avg_pit_stop_duration nulls: 347
   grid  avg_pit_stop_duration  constructorId  circuitId        era
0     3                    NaN              6          3  2010-2013
1     2                    NaN              6          3  2010-2013
2     4                    NaN              1          3  2010-2013
3     1                    NaN              9          3  2010-2013
4     5                    NaN            131          3  2010-2013


In [5]:
# Missing avg_pit_stop_duration is entirely 2010 (this dataset has no 2010 pit data).
# This is structural missingness, not random. Filling it would fabricate pit crew
# speed for a season that has none, which would corrupt the strategy signal I am
# measuring. So I drop these rows rather than impute.
# I also drop now, before the split, so every model (baseline, simple, tuned) trains 
# and tests on the exact same rows and the RMSE comparison stays apples-to-apples.

model_df = model_df.dropna(subset=['avg_pit_stop_duration']).copy()
print('rows after dropping 2010 (no pit data):', len(model_df))  # expect 4990

rows after dropping 2010 (no pit data): 4990


In [6]:
# How many unique categories does each categorical feature have?
# This decides whether one-hot encoding is reasonable or blows up the feature count.
for col in ['era', 'constructorId', 'circuitId']:
    print(f"{col}: {model_df[col].nunique()} unique values")

era: 4 unique values
constructorId: 23 unique values
circuitId: 35 unique values


In [7]:
# One-hot encode the three categorical features.
# constructorId and circuitId are ID integers (name tags, not quantities), and era
# is a text label. All three would mislead a model if left as-is, so each category
# becomes its own 0/1 column. No fake ordering, no leakage.
model_encoded = pd.get_dummies(
    model_df,
    columns=['constructorId', 'circuitId', 'era'],
    drop_first=False
)

# Build X (features) and y (target) from the encoded frame.
# Start from the numeric features that survived as-is, then add every one-hot column.
base_numeric = ['grid', 'avg_pit_stop_duration']
encoded_cols = [c for c in model_encoded.columns
                if c.startswith(('constructorId_', 'circuitId_', 'era_'))]

X = model_encoded[base_numeric + encoded_cols]
y = model_encoded['position']

print('X shape:', X.shape)   # rows should stay 4990, columns jump to ~50
print('y shape:', y.shape)

X shape: (4990, 64)
y shape: (4990,)


In [8]:
# confirm X contains ONLY legal features.
# No target, no ID columns, no leakage features, nothing non-numeric.
print('any non-numeric columns in X?', (X.dtypes == 'object').any())
print()
# these should NOT appear anywhere in X:
banned = ['position', 'position_change', 'positionText', 'points',
          'raceId', 'driverId', 'statusId', 'year']
leaked = [c for c in X.columns if c in banned]
print('banned columns found in X:', leaked)   # gotta be []
print()
print('the two numeric features present?',
      'grid' in X.columns, 'avg_pit_stop_duration' in X.columns)

any non-numeric columns in X? False

banned columns found in X: []

the two numeric features present? True True


In [9]:
# Split BEFORE any model exists. The test set is held back as unseen "new races"
# and is only used to score final models, never to train them.
# random_state fixes the split so the notebook is reproducible (same seed idea as
# the bootstrap in notebook 04).
X_train, X_test, y_train, y_test = train_test_split(X, y,
    test_size=0.2,       # 80% train, 20% test
    random_state=42
)
    
print('train rows:', len(X_train))   # expect ~3992
print('test rows:', len(X_test))     # expect ~998
print('train + test =', len(X_train) + len(X_test), 'of', len(X))

train rows: 3992
test rows: 998
train + test = 4990 of 4990


In [10]:

# BASELINE: the dumbest possible model. It ignores every feature and just predicts
# the mean finishing position for every single race. This is the bar every real
# model must beat. If a real model can't beat "always guess the average," something
# is wrong.
baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)

# Predict on the held-out test set and score with RMSE (root mean squared error).
# RMSE is in the same units as the target, so it reads as "on average, this many
# finishing positions off."
baseline_preds = baseline.predict(X_test)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_preds))

print('Baseline predicts the same number every time:', round(baseline.constant_[0][0], 2))
print('Baseline RMSE:', round(baseline_rmse, 3), 'positions')

Baseline predicts the same number every time: 9.42
Baseline RMSE: 5.405 positions


### Insight for baseline model
9.42 is the average finishing position across my training data. The dummy model ignores every feature and just guesses "9.42" for every single race, whether the driver started on pole or dead last. That's the middle-ish of the 1-to-20 range, which makes sense.


RMSE 5.405 means the dumb guess earns an error score of about 5.4 in finishing-position units. RMSE is not a plain average of misses: it squares errors before averaging, so big misses are penalized more heavily than small ones. Lower is better. That's my bar. It's the "I know nothing about this race" error. Every real model from here has to beat 5.405, or it's worthless. Now I have a number to beat, which is the entire point of a baseline.


### Simple Model: Decision Tree

**What I did**
- I built a Decision Tree to predict finishing position, using just two features: grid (where the driver started) and pit stop duration (how fast their team's crew was that season).
- I kept it to two features on purpose. This is my "simple" model, and its job is to show how far a basic model gets before I bring in anything fancy.
- I set the tree's max_depth to 5, which limits how many questions the tree is allowed to ask before it makes a guess. A shallow tree learns broad patterns; a deep tree memorizes the training data.
- I scored it with RMSE on both the training data and the held-out test data, so I could compare the two.

**Why I built it this way**
- I chose grid as the first feature because my earlier correlation work showed it is by far my strongest single predictor of finishing position. If any one feature was going to carry a simple model, it was this one.
- I added pit stop duration as the second feature so the model has one driver-side signal (grid) and one strategy-side signal (pit speed), which fits my research question.
- I limited the depth because a Decision Tree with no limit will memorize the training data and then fall apart on new races. Before settling on depth 5, I tried several depths and watched what happened to the gap between training error and test error. With no limit, the tree scored much better on training data than on test data, which is the classic sign of memorizing. Depth 5 gave the best test score with almost no gap, so that is what I kept.

**What I found**
- Baseline (the dumb model) had a test RMSE of 5.405 positions.
- My Decision Tree had a training RMSE of 3.286 and a test RMSE of 3.347.
- The gap between training and test error was only 0.061, which is tiny.

**What it means in plain English**
- My simple model scores an RMSE of about 3.3 finishing-position units, compared to about 5.4 for the dumb baseline. RMSE squares errors before averaging, so it punishes big misses more heavily, and lower is better. So even a basic two-feature model is a big improvement, it cuts the error by roughly two positions.
- The tiny gap between training and test error means the model is not overfitting. It performs about the same on races it trained on and races it has never seen, which means it learned real patterns instead of memorizing.
- The fact that just grid and pit speed already get this close tells me grid is doing most of the heavy lifting, exactly what my earlier analysis suggested. This sets up the tuned model, where the real question is how much the extra features add on top of what grid already explains.

**Caveats (limitations to keep in mind)**
- RMSE of 3.3 positions is a real improvement but not pinpoint accuracy. This model is a useful measuring tool, not a race predictor, and I am treating it that way.
- A single Decision Tree can be a little unstable: small changes in the data can change its splits. My tuned models (Random Forest, Gradient Boosting) address this by combining many trees, which is one reason I do not stop at this simple model.
- This model uses only two features, so it cannot yet tell me how much constructor, circuit, or era matter. That comparison is the job of the tuned model.

In [11]:
from sklearn.tree import DecisionTreeRegressor

# SIMPLE MODEL: a shallow Decision Tree on just two clean numeric features,
# grid (my strongest predictor) and pit stop duration (the strategy side).
# max_depth=5 keeps the tree shallow so it learns patterns instead of memorizing.
# I check train RMSE vs test RMSE: if they are close, the model is not overfitting.

simple_features = ['grid', 'avg_pit_stop_duration']
X_train_simple = X_train[simple_features]
X_test_simple  = X_test[simple_features]

simple = DecisionTreeRegressor(max_depth=5, random_state=42)
simple.fit(X_train_simple, y_train)

train_rmse = np.sqrt(mean_squared_error(y_train, simple.predict(X_train_simple)))
test_rmse  = np.sqrt(mean_squared_error(y_test,  simple.predict(X_test_simple)))

print('Baseline test RMSE:', 5.405)
print('Simple model train RMSE:', round(train_rmse, 3))
print('Simple model test RMSE: ', round(test_rmse, 3))
print('Overfit gap (test - train):', round(test_rmse - train_rmse, 3))

Baseline test RMSE: 5.405
Simple model train RMSE: 3.286
Simple model test RMSE:  3.347
Overfit gap (test - train): 0.061


In [12]:
# DEPTH CHECK: evidence for choosing max_depth=5.
# I try several depths and watch the gap between train and test RMSE.
# A big gap means the tree memorized the training data (overfitting).
for depth in [3, 5, 8, None]:
    t = DecisionTreeRegressor(max_depth=depth, random_state=42)
    t.fit(X_train_simple, y_train)
    tr = np.sqrt(mean_squared_error(y_train, t.predict(X_train_simple)))
    te = np.sqrt(mean_squared_error(y_test,  t.predict(X_test_simple)))
    print(f'depth={str(depth):4s} | train RMSE {tr:.3f} | test RMSE {te:.3f} | gap {te-tr:+.3f}')


depth=3    | train RMSE 3.445 | test RMSE 3.426 | gap -0.019
depth=5    | train RMSE 3.286 | test RMSE 3.347 | gap +0.061
depth=8    | train RMSE 3.025 | test RMSE 3.488 | gap +0.463
depth=None | train RMSE 2.191 | test RMSE 3.835 | gap +1.645


In [13]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# MODEL SELECTION BY CROSS-VALIDATION (training data only).
# Choosing between Random Forest and Gradient Boosting is a modeling decision,
# so it must not touch the test set. I compare both using 5-fold CV on the
# training set. The test set stays locked for final scoring only.

selection_candidates = {
    'Random Forest': RandomForestRegressor(
        n_estimators=300, max_depth=10, random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=300, max_depth=3, learning_rate=0.1, random_state=42
    ),
}

for name, model in selection_candidates.items():
    cv_scores = cross_val_score(
        model, X_train, y_train,
        cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1
    )
    print(f'{name}: CV RMSE = {round(-cv_scores.mean(), 3)} '
          f'(+/- {round(cv_scores.std(), 3)})')

Random Forest: CV RMSE = 3.123 (+/- 0.106)
Gradient Boosting: CV RMSE = 3.066 (+/- 0.11)


In [14]:
# ENSEMBLE MODELS ON ALL FEATURES: train/test RMSE for both candidates.
# The selection between them was already made by cross-validation above
# (training data only). These test-set numbers are reported for transparency.
# Prediction made before running: only a modest gain over the simple
# model's 3.347, because grid already explains most of the finish (r ~ 0.75).

candidates = {
    'Random Forest': RandomForestRegressor(
        n_estimators=300, max_depth=10, random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=300, max_depth=3, learning_rate=0.1, random_state=42
    ),
}

for name, model in candidates.items():
    model.fit(X_train, y_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
    test_rmse  = np.sqrt(mean_squared_error(y_test,  model.predict(X_test)))
    print(f'{name}:')
    print(f'  Train RMSE: {round(train_rmse, 3)}')
    print(f'  Test RMSE:  {round(test_rmse, 3)}')
    print(f'  Overfit gap (test - train): {round(test_rmse - train_rmse, 3)}')
    print()

print('Simple model test RMSE to beat: 3.347')

Random Forest:
  Train RMSE: 2.458
  Test RMSE:  3.078
  Overfit gap (test - train): 0.62

Gradient Boosting:
  Train RMSE: 2.691
  Test RMSE:  3.06
  Overfit gap (test - train): 0.369

Simple model test RMSE to beat: 3.347


## Tuned Model: Gradient Boosting with GridSearchCV

**What I did**
- I auditioned two ensemble models on all 64 feature columns: Random Forest and Gradient Boosting. Gradient Boosting won on both test RMSE (3.06 vs 3.078) and overfit gap (0.369 vs 0.62), so it advanced to tuning.
- The choice between the two was made using 5-fold cross validation on the training set only (Gradient Boosting 3.066 vs Random Forest 3.123), so the model selection decision never touched the test set. The test-set numbers for both are reported for transparency, but the selection was made by CV.
- I wrapped Gradient Boosting in GridSearchCV to test 8 combinations of hyperparameters: n_estimators (100 or 300), max_depth (3 or 5), and learning_rate (0.05 or 0.1).
- GridSearchCV used 5-fold cross validation on the training set only. It slices the training data into 5 chunks, trains on 4, scores on the held out 5th, rotates so every chunk gets a turn, and averages the scores. The test set stayed locked away the entire time and was only used once, at the end, to score the final winner.
- Before running, I predicted the tuned model would only modestly beat the simple model's 3.347, because my correlation analysis in notebook 04 already showed grid explains most of the finish (r of about 0.75).

**What I found**
- Best parameters: learning_rate 0.1, max_depth 3, n_estimators 300.
- Tuned test RMSE: 3.06, identical to the untuned audition run. GridSearch confirmed the settings I started with were already the best combo in the grid.
- Cross validation RMSE (3.066) and test RMSE (3.06) were nearly identical, which means the practice scores predicted the final exam almost perfectly. That is the mark of a trustworthy model.
- Overfit gap of 0.369, modest and acceptable.

**What it means in plain English**
- Tuning did not magically improve the model, and that is fine. What tuning bought me is proof: my hyperparameters are now validated by cross validation instead of guessed.
- My prediction came true. The fancy model with every feature and 300 boosted trees beat the simple 2-feature tree by only 0.29 positions. That small gain is not a failure, it is the finding: grid was already doing most of the work, so there was very little room left to improve.

**Caveats**
- I only searched 8 hyperparameter combinations. A wider grid might squeeze out a slightly better score, but given how little room is left above grid, I would expect diminishing returns.
- RMSE of 3.06 is an error score of about 3 finishing-position units, with big misses penalized more heavily. Races contain crashes, safety cars, and first lap chaos that no pre-race feature can see, so some error is irreducible.


In [15]:
from sklearn.model_selection import GridSearchCV

# TUNED MODEL, ROUND 2: GridSearchCV on the winner (Gradient Boosting).
# cv=5 means hyperparameters are picked using cross-validation on the
# TRAINING set only. The test set stays locked until the final score.

param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
}

grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)

grid.fit(X_train, y_train)

best_gb = grid.best_estimator_

train_rmse = np.sqrt(mean_squared_error(y_train, best_gb.predict(X_train)))
test_rmse  = np.sqrt(mean_squared_error(y_test,  best_gb.predict(X_test)))

print('Best parameters:', grid.best_params_)
print('Best CV RMSE (train folds):', round(-grid.best_score_, 3))
print('Tuned train RMSE:', round(train_rmse, 3))
print('Tuned test RMSE: ', round(test_rmse, 3))
print('Overfit gap (test - train):', round(test_rmse - train_rmse, 3))
print()
print('Untuned GB test RMSE: 3.06')
print('Simple model test RMSE: 3.347')
print('Baseline test RMSE: 5.405')

Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}
Best CV RMSE (train folds): 3.066
Tuned train RMSE: 2.691
Tuned test RMSE:  3.06
Overfit gap (test - train): 0.369

Untuned GB test RMSE: 3.06
Simple model test RMSE: 3.347
Baseline test RMSE: 5.405


## Model Comparison: The Whole Story in One Table

**What I did**
- I lined up all three models against the same locked test set: the baseline (always guesses the mean finish of 9.42), the simple decision tree (grid and pit stop duration only), and the tuned Gradient Boosting model (all features).
- Because every model was scored on the exact same 998 unseen test rows, the RMSE numbers are directly comparable.

**What I found**
- Baseline: 5.405. Simple: 3.347. Tuned: 3.060.
- Adding grid and pit stops to the baseline improved RMSE by about 2 full positions.
- Adding everything else (constructor, circuit, era, ensemble methods, tuning) improved it by only 0.29 more.

**What it means in plain English**
- Knowing where a driver qualified is worth about 2 finishing positions of prediction accuracy. Knowing everything else I am allowed to know before the race is worth about a quarter of a position on top of that.
- The gains shrink at each step because the most informative feature entered first. Grid ate most of the meal, and the remaining features fought over the leftovers.

**Caveats**
- This ordering reflects my feature set, not the sport itself. Features I scoped out (like separating qualifying pace from race pace) might change how the credit splits.

In [16]:
import pandas as pd

# MODEL COMPARISON: the whole modeling story in one table.
comparison = pd.DataFrame({
    'Model': ['Baseline (mean)', 'Simple (Decision Tree, 2 features)',
              'Tuned (Gradient Boosting, all features)'],
    'Test RMSE': [5.405, 3.347, 3.060],
    'Gain vs previous': ['-', '-2.058', '-0.287'],
})

comparison

,Model,Test RMSE,Gain vs previous
0,Baseline (mean),5.405,-
1,"Simple (Decision Tree, 2 features)",3.347,-2.058
2,"Tuned (Gradient Boosting, all features)",3.060,-0.287


## Feature Importance: Sizing the Levers

**What I did**
- I pulled feature importances from the tuned Gradient Boosting model. Importance measures how much each column contributed to the model's predictions, and all importances sum to 1.
- The model sees 64 columns, but they belong to only 5 real world features, because one hot encoding split constructor, circuit, and era into many 0/1 columns. I summed each one hot group back to its parent feature so the answer reads in stakeholder language.

**What I found**
- grid: 0.769
- constructor: 0.091
- avg_pit_stop_duration: 0.080
- circuit: 0.033
- era: 0.027

**What it means in plain English**
- Grid dominated the model's predictive signal, at 0.769 of total importance. Where you start is most of where you finish. These importances measure how much the model used each feature to reduce error, they are predictive, not causal.
- The team side levers, constructor identity and pit stop speed, combine for about 17 percent. They are real and measurable, but a fraction of grid.
- This echoes my statistical analysis. Notebook 04 found a grid to finish correlation of about 0.75, and the model, built by a completely different method, assigned grid an importance of 0.769. Two independent approaches landed on the same answer, which is exactly what I predicted before running the model.
- The stakeholder takeaway: once you know where a driver qualified, everything the team controls on race day moves the needle by a combined 17 percent. Qualifying pace dominates, and pit execution is a real but secondary lever.

**Caveats**
- Importance measures predictive usefulness inside this model, not physical cause. Grid partly absorbs car quality, because fast cars qualify well and finish well, so some of the constructor signal is hiding inside grid.
- Importances are specific to this feature set and this model. A different encoding or model family could shift the exact numbers, though I would not expect the ranking to change.
- Impurity based importance can favor continuous features like grid over one hot 0/1 columns, because a continuous feature offers more possible split points. Permutation importance would be a stronger check and is noted as future work.

In [17]:
# FEATURE IMPORTANCE: how much did each real-world feature contribute?
# The model has 64 columns, but they belong to 5 real features.
# I sum each one-hot group back to its parent so the answer reads
# in stakeholder language: grid vs pit stops vs constructor vs circuit vs era.

importances = pd.Series(best_gb.feature_importances_, index=X_train.columns)

def group_importance(col_name):
    if col_name.startswith('constructorId_'):
        return 'constructor'
    elif col_name.startswith('circuitId_'):
        return 'circuit'
    elif col_name.startswith('era_'):
        return 'era'
    else:
        return col_name  # grid, avg_pit_stop_duration

grouped = importances.groupby(group_importance).sum().sort_values(ascending=False)

print('Feature importance (grouped, sums to 1.0):')
for feature, score in grouped.items():
    print(f'  {feature}: {round(score, 3)}')

Feature importance (grouped, sums to 1.0):
  grid: 0.769
  constructor: 0.091
  avg_pit_stop_duration: 0.08
  circuit: 0.033
  era: 0.027


## Sanity Check: Where Does the Model Make Its Mistakes?

**What I did**
- I calculated residuals on the test set. A residual is actual finish minus predicted finish, so positive means the driver finished worse than predicted and negative means better.
- I bucketed the residuals by grid position (P1 to P5, P6 to P10, P11 to P15, P16 and back) to check whether the model is systematically biased against any part of the field.
- The mean residual per bucket is the bias meter: near zero means no systematic bias. The standard deviation is the chaos meter: it shows how unpredictable each group is.

**What I found**
- Mean residuals were near zero in all four buckets (largest was +0.339 for P6 to P10). No part of the grid is systematically shortchanged.
- The standard deviations told a story I did not predict. I expected the front and back of the grid to be the wildest. Instead the midfield was the most unpredictable (std of about 3.5 for P6 to P10 and 3.2 for P11 to P15), while the front (2.73) and the back (2.81) were the calmest.

**What it means in plain English**
- The model passes the bias check. Its errors are noise, not a systematic lean against front runners or backmarkers.
- The midfield result makes racing sense once you see it. A car starting P2 usually finishes near the front and a car starting P18 usually finishes near the back, but a P8 car can grab a podium on a well timed safety car or get taken out at turn one. The midfield is where strategy, traffic, and luck collide.
- This matters directly for my stakeholder, who works at a midfield team: the model is least certain exactly where they race. Midfield outcomes are the hardest to predict, which is honest and useful to say out loud.

**Caveats**
- I predicted the front and back would be noisiest and the data corrected me. I am keeping that in the notebook rather than hiding it, because testing a hunch and being wrong is part of the process.
- Bucket boundaries are my choice. Different cut points would shift the exact numbers, though the midfield pattern is strong enough that I would not expect it to disappear.

In [18]:
# RESIDUAL CHECK: where does the model make its mistakes?
# Residual = actual - predicted. Positive = driver finished worse than
# predicted, negative = better. I bucket by grid position to see if the
# model is biased against any part of the field.

residuals = y_test - best_gb.predict(X_test)

resid_df = pd.DataFrame({
    'grid': X_test['grid'],
    'residual': residuals,
})

resid_df['grid_bucket'] = pd.cut(
    resid_df['grid'],
    bins=[0, 5, 10, 15, 25],
    labels=['P1-P5', 'P6-P10', 'P11-P15', 'P16+'],
)

print(resid_df.groupby('grid_bucket', observed=True)['residual']
      .agg(['mean', 'std', 'count']).round(3))

              mean    std  count
grid_bucket                     
P1-P5       -0.047  2.730    273
P6-P10       0.339  3.496    229
P11-P15      0.084  3.234    238
P16+         0.053  2.805    258


## Final Model: Selection and Justification

**What I did**
- I compared all three models on test RMSE, overfit gap, and usefulness to the stakeholder question, then selected one final model.

**What I found**
- The tuned Gradient Boosting model is my final model: test RMSE 3.06, overfit gap 0.369, hyperparameters validated by 5-fold cross validation, and clean residual checks.

**What it means in plain English**
- I chose the tuned model for two reasons. It is the most accurate on unseen races, and more importantly it is the model that produces the feature importance breakdown, which is the actual answer my stakeholder needs. The point of this notebook was never to predict race results like a crystal ball. It was to measure the size of the levers a mid tier team is choosing between.
- The simple decision tree stays in the notebook on purpose. It is evidence that grid alone does most of the work, and the tiny gap between the simple and tuned models is itself a finding.
- The honest summary: this model is a measuring instrument, not a fortune teller. It confirmed and quantified what my correlation analysis suggested. Qualifying pace dominates race outcomes, and pit stop execution is a real but secondary lever.

**Caveats**
- An RMSE of 3.06 positions is far too coarse for race day decisions. This model informs where a team should invest, not what to do on lap 30.
- The 2010 season is absent from the modeling frame because the dataset has no 2010 pit stop data. I dropped those 347 rows rather than fill them, because inventing pit stop values would fabricate the exact signal I was measuring.
- Modeling is the thinnest new insight in this project by design. The statistical analysis in notebook 04 did the intellectual heavy lifting, and this notebook confirms and quantifies it.
- avg_pit_stop_duration is averaged per constructor per season, so a row from early 2023 carries information from races later that season. This is temporal leakage if the model is read as a live pre-race predictor, which is why I frame it as a retrospective measuring instrument instead. A future version would use previous-season or rolling prior-race pit averages, at the cost of losing the 2011 season (its prior year, 2010, has no pit data).
- The random 80/20 split means training and test rows share seasons. A stricter design for time ordered data is a temporal split, training on earlier years and testing on later ones. Noted as future work alongside the pit stop fix.